> **说明**：本 notebook 是 [langchain-ai/langgraph-101](https://github.com/langchain-ai/langgraph-101) 的**简体中文翻译版**，供 Google Colab / Jupyter 使用。
>
> - 原文仓库：<https://github.com/langchain-ai/langgraph-101>
> - 建议学习顺序：`101_langchain_langgraph` → `102_middleware` → `deep_agents` → `deep_agents_extra`
> - **代码单元格保持英文原样**（标识符、字符串、prompt 均未改动）；仅翻译 Markdown 与 `#` 注释。
> - 图片路径仍相对原仓库；在 Colab 中请先 clone 原仓库或自行调整图片路径。


# LangGraph 进阶概念：中间件（Middleware）与人机协同（HITL）

欢迎来到 LangGraph 进阶概念！本 notebook 建立在 LangGraph 101 的基础上，介绍两种面向生产智能体的强大模式。

**你将学到：**
- **人机协同（Human-in-the-Loop, HITL）** —— 暂停智能体以供人工审核与批准
- **中间件（Middleware）** —— 在执行关键节点修改智能体行为
- **工具审核（Tool Review）** —— 为敏感工具加入审批工作流
- **动态行为** —— 根据上下文调整智能体响应

**前置条件：** 请先完成 `langgraph_101.ipynb`（本系列中的 `101_langchain_langgraph`）
</br>
</br>

---
</br>

> **说明：** 这些模式对生产智能体至关重要 —— 尤其在安全、合规与用户可控性要求较高时。


## 环境准备

快速配置环境。


In [ ]:
# 将项目根目录加入 Python 路径，以便从 utils 模块导入
import sys
from pathlib import Path
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 从集中式 utils 模块导入 model
# 避免各 notebook 重复代码，并使用一致的模型配置
from utils.models import model

# 备选：若想在本处内联定义模型而非使用集中配置，取消下面注释：
# from dotenv import load_dotenv
# load_dotenv(dotenv_path="../.env", override=True)
# from langchain.chat_models import init_chat_model
# model = init_chat_model("openai:o3-mini")

# 说明：若使用其他供应商（Azure、Bedrock、Vertex AI），请修改 utils/models.py
# 切换 LLM 供应商的详细说明见 utils/models.py

## 第 1 部分：用中断（Interrupts）实现人机协同

### 问题

假设你在构建能发邮件或下单的智能体。你不希望这些动作自动执行 —— 而是希望先经人工批准！

**人机协同（HITL）** 可以让你：
- 暂停执行以供审核
- 批准、拒绝或编辑动作
- 为敏感操作加入安全控制

### 工作原理

1. 智能体遇到 `interrupt()` —— 执行暂停
2. 系统向人类展示相关信息
3. 人类提供输入（批准 / 拒绝 / 编辑）
4. 智能体通过 `Command(resume=...)` 恢复执行


### 示例 1：简单审批工作流

从一个简单例子开始 —— 在发送邮件前请求批准。


In [ ]:
from langgraph.types import interrupt
from langchain_core.tools import tool

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    
    # 暂停以等待人工批准
    approval = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "Do you want to send this email?"
    })
    
    if approval.get("approved"): # Will be true if accepted, false if declined
        # 生产环境中这里会真正发送邮件
        return f" Email sent to {to} with subject '{subject}'"
    else:
        return "Email cancelled by user"

# 直接测试工具
print("Tool created successfully!")
print(f"Tool name: {send_email.name}")
print(f"Tool description: {send_email.description}")

### 创建带人机协同的智能体

现在创建使用该工具的智能体。**记住：** 中断需要检查点（checkpointer）！


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# 创建用于持久化的检查点（checkpointer）
checkpointer = MemorySaver()

# 创建带邮件工具的智能体
agent = create_agent(
    model=model,
    tools=[send_email],
    system_prompt="You are a helpful email assistant. When asked to send emails, use the send_email tool.",
    checkpointer=checkpointer  # Required for interrupts
)

### 运行到中断处

运行智能体，观察它因审批而暂停：


In [ ]:
from langchain.messages import HumanMessage
from langsmith import uuid7

# 为本次对话创建唯一线程
config = {"configurable": {"thread_id": uuid7()}}

# 运行智能体，观察其因审批而暂停
result = agent.invoke(
    {
        "messages": [HumanMessage(content="Send an email to alice@example.com with subject 'Meeting Tomorrow' and body 'Let's meet at 3pm.'")]
    },
    config=config
)

# 检查是否命中了中断（interrupt）

if "__interrupt__" in result:
    print("Agent paused for approval\n")

    interrupt_info = result["__interrupt__"][0]

    print("Interrupt details:")
    print(f"  To: {interrupt_info.value['to']}")
    print(f"  Subject: {interrupt_info.value['subject']}")
    print(f"  Body: {interrupt_info.value['body']}")
    print(f"  Message: {interrupt_info.value['message']}")
else:
    print("Agent completed without interrupt")

### 批准后恢复

现在批准这封邮件，让智能体继续：


In [ ]:
from langgraph.types import Command

# 批准后恢复
result = agent.invoke(
    Command(resume={"approved": True}),
    config=config
)

# 打印最终回复
print("Final response:")
print(result["messages"][-1].content)

### 练习：试着拒绝邮件

再跑一遍单元格，但这次传入 `{"approved": False}` 来拒绝邮件：


In [ ]:
# 拒绝示例使用新线程
config_2 = {"configurable": {"thread_id": uuid7()}}

# 运行直到中断
result = agent.invoke(
    {
        "messages": [HumanMessage(content="Send an email to bob@example.com saying 'Hello!'")]
    },
    config=config_2
)

# 拒绝后恢复
result = agent.invoke(
    Command(resume={"approved": False}),  # Reject the email
    config=config_2
)

print("Final response:")
print(result["messages"][-1].content)

## 第 2 部分：进阶模式 —— 执行前编辑

有时你想**编辑**工具调用，而不仅仅是批准 / 拒绝。增强一下我们的工具：


In [ ]:
@tool
def send_email_v2(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    
    # 暂停以供人工审核
    response = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "Review this email. You can approve, reject, or edit it."
    })
    
    # 处理不同类型的响应
    if response["type"] == "approve":
        return f"Email sent to {to} with subject '{subject}'"

    elif response["type"] == "reject":
        return "Email cancelled"

    elif response["type"] == "edit":
        # 使用编辑后的值
        to = response.get("to", to)
        subject = response.get("subject", subject)
        body = response.get("body", body)
        return f"""Email sent with edits:
                To: {to}
                Subject: {subject}
                Body: {body}"""
    
    return "Unknown response"

# 用增强后的工具创建新智能体
agent_v2 = create_agent(
    model=model,
    tools=[send_email_v2],
    system_prompt="You are a helpful email assistant.",
    checkpointer=MemorySaver()
)

In [ ]:
# 运行并编辑邮件
config_3 = {"configurable": {"thread_id": uuid7()}}

# 运行直到中断
result = agent_v2.invoke(
    {
        "messages": [HumanMessage(content="Send an email to team@example.com about the meeting")]
    },
    config=config_3
)

print("Paused for review...\n")

现在把邮件主题改成 URGENT meeting！


In [ ]:
# 带编辑结果恢复
result = agent_v2.invoke(
    Command(resume={
        "type": "edit",
        "subject": "URGENT: Meeting Today at 2pm",  # We have edited the email subject
        "body": "This is the edited email body with more details."
    }),
    config=config_3
)

print("Final response:")
print(result["messages"][-1].content)

## 第 3 部分：中间件（Middleware）入门

**中间件（Middleware）** 提供对智能体循环的细粒度控制。它可以让你：
- 在模型调用前后检查状态
- 动态修改模型请求
- 在关键执行点加入自定义逻辑

### 智能体循环

```
Input --> [before_model] --> [wrap_model_call] --> Model --> [after_model] --> Tools --> ...
```

中间件挂接到该循环：
- **`before_model`** —— 在模型执行前运行，可更新状态
- **`wrap_model_call`** —— 包裹模型调用，控制何时 / 如何调用模型
- **`after_model`** —— 在模型执行后、工具之前运行

### 两种 Hook 风格

**节点风格（Node-style）hooks** 顺序执行：
- `before_agent`、`before_model`、`after_model`、`after_agent`
- 适合日志、校验、状态更新

**包裹风格（Wrap-style）hooks** 拦截执行：
- `wrap_model_call`、`wrap_tool_call`
- 对 handler 调用有完全控制权
- 适合重试、缓存、转换


### 示例 1：动态系统提示（System Prompt）

创建根据用户角色改变系统提示的中间件：


In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from typing import TypedDict

# 定义上下文 schema
class Context(TypedDict):
    user_role: str

# 用装饰器创建中间件
@dynamic_prompt
def dynamic_prompt_middleware(request: ModelRequest) -> str:
    """Adjust system prompt based on user role."""
    
    user_role = request.runtime.context.get("user_role", "general")
    
    if user_role == "expert":
        return "You are an AI assistant for experts. Provide detailed technical responses with code examples."
    elif user_role == "beginner":
        return "You are an AI assistant for beginners. Explain concepts simply, avoid jargon."
    else:
        return "You are a helpful AI assistant."

In [ ]:
from langchain_core.tools import tool

@tool
def explain_concept(concept: str) -> str:
    """Explain a programming concept."""
    explanations = {
        "async": "Asynchronous programming allows code to run without blocking.",
        "recursion": "Recursion is when a function calls itself."
    }
    return explanations.get(concept.lower(), "Concept not found.")

# 创建带中间件的智能体
agent_with_middleware = create_agent(
    model=model,
    tools=[explain_concept],
    middleware=[dynamic_prompt_middleware],
    context_schema=Context
)

### 测试不同用户角色

看看智能体如何根据用户角色给出不同响应：


In [ ]:
# 专家用户
print("=" * 50)
print("EXPERT USER")
print("=" * 50)

result = agent_with_middleware.invoke(
    {"messages": [HumanMessage(content="Explain async programming")]},
    context={"user_role": "expert"}
)
print(result["messages"][-1].content)
print()

# 初学者用户
print("=" * 50)
print("BEGINNER USER")
print("=" * 50)

result = agent_with_middleware.invoke(
    {"messages": [HumanMessage(content="Explain async programming")]},
    context={"user_role": "beginner"}
)
print(result["messages"][-1].content)

### 示例 2：自定义中间件 —— 请求日志器

中间件让你挂入智能体循环，观察每一步发生了什么。这对调试与理解智能体行为非常有用。

**智能体循环：**
User Input --> [before_model] --> [wrap_model_call] --> Model --> [after_model] --> Tools --> ...

**我们将构建：**
一个在每一步打印信息的日志器：
- **模型前（Before model）** —— 对话中有多少条消息？
- **包裹模型调用（Wrap model call）** —— 使用了哪个模型与哪些工具？
- **模型后（After model）** —— 模型是调用了工具还是给出了最终答案？

这类似加调试用的 `print()`，但更干净、可复用！

创建用于调试、记录模型请求的中间件：


In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, ModelRequest, ModelResponse
from typing import Any, Callable

class RequestLoggerMiddleware(AgentMiddleware):
    """Logs all model requests for debugging."""
    
    def before_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        """Log before model execution."""
        message_count = len(state.get("messages", []))
        print(f"[BEFORE MODEL] Processing {message_count} messages")
        return None  # Don't modify state
    
    def wrap_model_call(
        self, 
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
        """Log the model request details and call the handler."""
        print(f"  [MODEL REQUEST]")
        print(f"   Model: {request.model if hasattr(request, 'model') else 'default'}")
        print(f"   Tools available: {len(request.tools) if request.tools else 0}")
        
        # 调用实际的模型 handler
        return handler(request)
    
    def after_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        """Log after model execution."""
        last_message = state["messages"][-1]
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f" [AFTER MODEL] Model requested {len(last_message.tool_calls)} tool call(s)")
        else:
            print(f" [AFTER MODEL] Model provided final response")
        return None  # Don't modify state

In [ ]:
# 创建带日志中间件的智能体
agent_with_logger = create_agent(
    model=model,
    tools=[explain_concept],
    middleware=[RequestLoggerMiddleware()],
)

### 预期现象

带日志器运行智能体时，你会实时看到执行流程：

**第一轮：**
1. `[BEFORE MODEL]` —— 显示起始消息数量
2. `[MODEL REQUEST]` —— 显示可用模型与工具（来自 wrap_model_call）
3. `[AFTER MODEL]` —— 模型决定调用 `explain_concept` 工具

**第二轮（工具执行后）：**
1. `[BEFORE MODEL]` —— 消息更多了（含工具结果）
2. `[MODEL REQUEST]` —— 再次打印模型信息
3. `[AFTER MODEL]` —— 模型给出最终答案（不再需要工具）

这能让你细致观察智能体的决策过程。

运行一下：


In [ ]:
# 运行并观察日志
print("\n" + "=" * 50)
print("RUNNING AGENT WITH LOGGER")
print("=" * 50 + "\n")

result = agent_with_logger.invoke({
    "messages": [{"role": "user", "content": "Explain recursion"}]
})

print("\n" + "=" * 50)
print("FINAL RESPONSE")
print("=" * 50)
print(result["messages"][-1].content)

## 第 4 部分：组合中间件与人机协同

把人机协同与中间件结合起来，做成更接近生产的智能体：


In [ ]:
# 需要批准的敏感工具
@tool
def delete_database(database_name: str) -> str:
    """Delete a database. THIS IS DANGEROUS!"""
    
    response = interrupt({
        "action": "delete_database",
        "database_name": database_name,
        "warning": "This will permanently delete the database!",
        "message": "Are you absolutely sure?"
    })
    
    if response.get("confirmed"):
        return f"Database '{database_name}' has been deleted (simulation)"
    else:
        return "Database deletion cancelled"

# 用于追踪危险操作的中间件
class SafetyMiddleware(AgentMiddleware):
    """Add safety checks and logging."""
    
    name = "safety_checker"
    
    def after_model(self, state: AgentState) -> dict[str, Any] | None:
        """Check for dangerous tool calls."""
        last_message = state["messages"][-1]
        
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            for tool_call in last_message.tool_calls:
                if "delete" in tool_call["name"].lower():
                    print("   [SAFETY] Dangerous operation detected!")
                    print(f"   Tool: {tool_call['name']}")
                    print(f"   Args: {tool_call['args']}")
        
        return None

# 创建生产级智能体
production_agent = create_agent(
    model=model,
    tools=[delete_database],
    middleware=[SafetyMiddleware()],
    checkpointer=MemorySaver()
)

  ### 预期现象：分层安全生效

  尝试危险操作时，你会看到**两层**安全机制同时激活：

  **第 1 层 —— 中间件检测：**
  - `[SAFETY] Dangerous operation detected!` —— 中间件发现删除操作
  - 记录工具名与参数，便于审计

  **第 2 层 —— 人工批准（中断）：**
  - 智能体在 `interrupt()` 处暂停
  - 向人工审核者展示警告
  - 在明确批准前不会继续执行

  **这就是纵深防御：** 中间件监控全部操作，中断则对关键动作强制人工批准。


In [ ]:
# 测试组合模式
config_4 = {"configurable": {"thread_id": uuid7()}}

print("\n" + "=" * 50)
print("DANGEROUS OPERATION ATTEMPT")
print("=" * 50 + "\n")

# 运行直到中断
result = production_agent.invoke(
    {
        "messages": [HumanMessage(content="Delete the production_db database")]
    },
    config=config_4
)

if "__interrupt__" in result:
    interrupt_info = result["__interrupt__"][0]
    print("\n  Human approval required:")
    print(f"   {interrupt_info.value['warning']}")
    print(f"   Database: {interrupt_info.value['database_name']}")

print("\n(In a real app, a human would review this before proceeding)")

## 要点总结

### 人机协同 / 中断（Interrupts）
- 用 `interrupt()` 暂停执行
- 需要 `checkpointer` 做持久化
- 用 `Command(resume=value)` 恢复
- 非常适合审批工作流与敏感操作

### 中间件（Middleware）
- **节点风格 hooks**：`before_model`、`after_model` —— 顺序逻辑、校验、日志
- **包裹风格 hooks**：`wrap_model_call`、`wrap_tool_call` —— 完全控制、重试、转换
- **装饰器**：`@dynamic_prompt`、`@before_model`、`@wrap_model_call` 快速写中间件
- **类**：继承 `AgentMiddleware` 构建复杂、可复用组件

### 何时用什么？

**适合用中断的情况：**
- 需要对动作做人工批准
- 想审核 / 编辑工具调用
- 需要校验用户输入

**适合用中间件的情况：**
- 需要动态修改智能体行为
- 想加日志 / 监控
- 需要强制策略（token 限制、安全检查）
- 想根据上下文个性化响应

**节点风格 vs 包裹风格：**
- 节点风格适合顺序操作（日志、校验）
- 包裹风格适合控制流（重试、回退、缓存）


## 练习（可选）

试着构建一个智能体：
1. 有一个下单 / 购买工具
2. 用中间件检查金额是否超过 $1000
3. 超过 $1000 时用中断要求批准
4. 低于 $1000 时自动处理

提示：把 `before_model` 中间件与条件性 `interrupt()` 逻辑结合起来！


In [ ]:
# 在此编写你的代码！
# 挑战：构建购买审批智能体

# @tool
# def make_purchase(item: str, amount: float) -> str:
# ...

# class PurchaseLimitMiddleware(AgentMiddleware):
# ...

## 下一步

你现在拥有了构建生产智能体的强大工具！

**继续学习：**
1.  查看 `multi_agent.ipynb` 了解多智能体系统
2.  探索内置中间件（Summarization、Anthropic Prompt Caching）
3.  为自己的场景编写自定义中间件
4.  用 LangSmith 做调试与监控

**资源：**
- [中间件文档](https://docs.langchain.com/oss/python/langchain/middleware)
- [人机协同指南](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangGraph 文档](https://langchain-ai.github.io/langgraph/)

</br>
</br>
